In [3]:
import os
import shutil
import random

original_data_path = "D:/VS Code/Coding/Deepfake_Project/Phase1_Preprocessing/processed_data"
test_sample_path = "D:/VS Code/Coding/Deepfake_Project/Phase2_Training/test_sample"

for category in ['real', 'fake']:
    os.makedirs(os.path.join(test_sample_path, category), exist_ok=True)
    
    source_folder = os.path.join(original_data_path, category)
    all_images = os.listdir(source_folder)
    
    sample_images = random.sample(all_images, 200)
    
    print(f"Copying 200 {category} images for testing...")
    for img_name in sample_images:
        shutil.copy(os.path.join(source_folder, img_name), 
                    os.path.join(test_sample_path, category, img_name))

print("Small Dataset test wala dataset", test_sample_path)

Copying 200 real images for testing...
Copying 200 fake images for testing...
Small Dataset test wala dataset D:/VS Code/Coding/Deepfake_Project/Phase2_Training/test_sample


In [1]:
import os
import sys
import tensorflow as tf

# 1. Block the messy Roaming folder pollution
os.environ['PYTHONNOUSERSITE'] = '1'

# 2. Automatically find and add the DLL directory for your active environment
env_bin = os.path.join(sys.prefix, 'Library', 'bin')
if os.path.exists(env_bin):
    os.add_dll_directory(env_bin)
    print(f"✅ Added DLL directory: {env_bin}")
else:
    print("ℹ️ DLL directory not found, skipping manual link.")

# 3. Final Check for the RTX A6000
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ SUCCESS: Found {len(gpus)} GPU(s).")
    print(f"Hardware: {tf.config.experimental.get_device_details(gpus[0])['device_name']}")
else:
    print("❌ GPU still not detected. We need to install the CUDA bridge.")

✅ Added DLL directory: c:\Users\gpu.arl\AppData\Local\miniconda3\envs\artifact_lens_env\Library\bin
✅ SUCCESS: Found 1 GPU(s).
Hardware: NVIDIA RTX A6000


In [1]:
import os
# This prevents the environment from looking at the broken Roaming folder
os.environ['PYTHONNOUSERSITE'] = '1' 

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')

if gpus:
    print(f"✅ SUCCESS: Found {len(gpus)} GPU(s).")
    print(f"Hardware: {tf.config.experimental.get_device_details(gpus[0])['device_name']}")
else:
    print("❌ GPU not found. Please restart the VS Code window.")

✅ SUCCESS: Found 1 GPU(s).
Hardware: NVIDIA RTX A6000


In [2]:
import os
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras import mixed_precision

# This uses 16-bit math to make training up to 3x faster on high-end NVIDIA GPUs
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)
IMG_SIZE = (224, 224)
BATCH_SIZE = 16 
EPOCHS = 20
BASE_PATH = "D:/rishi/Deepfake_Detection/Phase1_Preprocessing/processed_data"
CHECKPOINT_PATH = "D:/rishi/Deepfake_Detection/Phase2_Training/artifact_lens_final.h5"

train_datagen = ImageDataGenerator(rescale=1./255, rotation_range=20, horizontal_flip=True, validation_split=0.2)

train_generator = train_datagen.flow_from_directory(BASE_PATH, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary', subset='training')
val_generator = train_datagen.flow_from_directory(BASE_PATH, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode='binary', subset='validation')

# If power went out and want to RESUME, uncomment the line below:
# model = tf.keras.models.load_model(CHECKPOINT_PATH) 

def build_model():
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    base_model.trainable = False
    x = GlobalAveragePooling2D()(base_model.output)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.5)(x)
    # Add dtype='float32' here for mixed precision stability
    predictions = Dense(1, activation='sigmoid', dtype='float32')(x)
    return Model(inputs=base_model.input, outputs=predictions)

model = build_model() # Comment this out if you are resuming from a loaded model

model.compile(optimizer=Adam(learning_rate=0.0001), loss='binary_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1)

checkpoint = ModelCheckpoint(
    filepath=CHECKPOINT_PATH,
    monitor='val_accuracy',
    save_best_only=True,   
    mode='max',
    verbose=1
)

print("Starting Phase 2 with Checkpointing enabled...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=[early_stop, checkpoint], 
    verbose=2
)

model.save("D:/rishi/Deepfake_Detection/Phase2_Training/artifact_lens_final.h5")
print("Project ArtifactLens: Final training complete.")

INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA RTX A6000, compute capability 8.6
Found 110911 images belonging to 2 classes.
Found 27727 images belonging to 2 classes.
Starting Phase 2 with Checkpointing enabled...
Epoch 1/20

Epoch 1: val_accuracy improved from -inf to 0.59707, saving model to D:/rishi/Deepfake_Detection/Phase2_Training\artifact_lens_final.h5
6932/6932 - 923s - loss: 0.6846 - accuracy: 0.5587 - val_loss: 0.6645 - val_accuracy: 0.5971 - 923s/epoch - 133ms/step
Epoch 2/20

Epoch 2: val_accuracy improved from 0.59707 to 0.62069, saving model to D:/rishi/Deepfake_Detection/Phase2_Training\artifact_lens_final.h5
6932/6932 - 494s - loss: 0.6506 - accuracy: 0.6241 - val_loss: 0.6374 - val_accuracy: 0.6207 - 494s/epoch - 71ms/step
Epoch 3/20

Epoch 3: val_accuracy did not improve from 0.62069
6932/6932 - 494s - loss: 0.6261 -